# BTC 双均线策略回测（统一基线）

本 notebook 通过基线入口调用共享引擎 `strategies.research.run_backtest`，与脚本使用相同的回测逻辑。

- MA20 上穿 MA60 时买入，下穿时卖出；只做多，不做空。
- 在前一日收盘确认信号，下一日开盘成交。
- 单边手续费 0.1%，单边滑点 0.05%；策略与买入持有基准均计入双边成本。
- 排除快照最后一根未确认日线；期末仍持仓时按最后收盘价计入滑点和手续费结算。
- 初始资金 10,000 USD；前 61 根日线用于预热，回测区间由下方共享引擎输出。


In [ ]:
from pathlib import Path
import sys

# 从仓库根目录或 notebooks 目录运行均可定位共享引擎
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "strategies/research.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from strategies.ma_cross_backtest import baseline_result

result = baseline_result(root / "data/btc_usd_1d_raw.json")
summary = result["summary"]
print(f"回测区间: {summary['start']} ~ {summary['end']}，{summary['days']} 根已确认日线")
print(f"数据 SHA-256: {summary['data_sha256']}")


## 交易明细

以下逐笔显示信号日、实际成交日、成交价和扣除双边成本后的净收益。`forced` 标记期末结算，该笔不计入已完成交易胜率。


In [ ]:
trades = result["trades"]
curve = result["equity"]
closed_returns = [trade["return_pct"] for trade in trades if not trade["forced"]]
for trade in trades:
    print(f"信号 {trade['signal_date']} → 买入 {trade['entry_date']} @ {trade['entry_price']:,.2f}")
    kind = "期末结算" if trade["forced"] else "卖出"
    print(f"  {kind} {trade['exit_date']} @ {trade['exit_price']:,.2f}，净收益 {trade['return_pct']:+.2f}%")


## 结果统计

策略收益和同期买入持有均扣除手续费与滑点；胜率仅统计自然退出的已完成交易，最大回撤按每日收盘权益计算并包含初始资金和期末结算成本。


In [ ]:
final = summary["final"]
tot = summary["return_pct"]
bh = summary["buy_hold_pct"]
dd = summary["max_dd_pct"]
win_rate = summary["win_rate_pct"]
win_text = "无已完成交易" if win_rate is None else f"{win_rate:.1f}%"
print(f"初始 {summary['initial']:,.0f} USD → 期末 {final:,.2f} USD")
print(f"策略收益 {tot:+.2f}% ｜同期买入持有 {bh:+.2f}%")
print(f"完成交易 {summary['completed_trades']} 笔 ｜期末结算 {summary['terminal_liquidations']} 笔 ｜胜率 {win_text} ｜最大回撤 {dd:.2f}%")


## 中文解读

以上结果来自固定历史快照上的 MA20/60 基线。收益、回撤和交易次数直接取自统一引擎，不能据此保证未来收益。

参数筛选及样本外评估见 `docs/策略迭代报告.md`。比较改进方案时，应保持数据区间、手续费、滑点及成交时点一致。

本 notebook 仅用于离线研究，不会启动真实交易或修改持续观察中的模拟账户；回测结果仍需结合样本外检验和持续纸面观察评估。
